In [1]:
!pip install -q sentencepiece datasets

In [2]:
%%writefile args.py
from dataclasses import dataclass
@dataclass
class modelargs:
    dim:int=512
    n_layers:int=14
    n_heads:int=8
    n_kv_heads:int=1
    vocab_n:int=16384
    hidden_dim:int=1536
    max_seq_n:int=1024
    max_batch_n:int=32
    norm_eps:float=1e-5

Writing args.py


In [3]:
%%writefile cache.py
import torch
from args import modelargs
class KVcache:
    def __init__(self,a:modelargs,d:torch.device):
        """Initializes KV cache for fast generation."""
        self.hd=a.dim//a.n_heads
        self.kc=torch.zeros((a.max_batch_n,a.n_kv_heads,a.max_seq_n,self.hd),dtype=torch.float16,device=d)
        self.vc=torch.zeros((a.max_batch_n,a.n_kv_heads,a.max_seq_n,self.hd),dtype=torch.float16,device=d)
        self.sn=0
    def change(self,k:torch.Tensor,v:torch.Tensor)->tuple[torch.Tensor,torch.Tensor]:
        """Updates and returns cached keys and values."""
        bn,_,sn,_=k.shape
        k=k.to(dtype=torch.float16)
        v=v.to(dtype=torch.float16)
        self.kc[:bn,:,self.sn:self.sn+sn,:]=k
        self.vc[:bn,:,self.sn:self.sn+sn,:]=v
        self.sn+=sn
        return (self.kc[:bn,:,:self.sn,:],self.vc[:bn,:,:self.sn,:])

Writing cache.py


In [4]:
%%writefile rope.py
import torch
import torch.nn as nn
from args import modelargs
class RoPE(nn.Module):
    def __init__(self,d,a:modelargs,tp=10000.0):
        """Initializes Rotary Positional Embeddings."""
        super().__init__()
        self.d=d
        ti=1.0/(tp**(torch.arange(0,d,2).float()/d))
        self.register_buffer("ti",ti)
        self.msn=a.max_seq_n
        self._bc(a.max_seq_n)
    def _bc(self,sn):
        """Builds cosine and sine caches."""
        t=torch.arange(sn,dtype=self.ti.dtype,device=self.ti.device)
        te=t.unsqueeze(1)*self.ti.unsqueeze(0)
        f=torch.cat((te,te),dim=-1)
        self.register_buffer("cc",f.cos()[None,None,:,:])
        self.register_buffer("sc",f.sin()[None,None,:,:])
    def forward(self,i,sn):
        """Returns cached cosine and sine values."""
        if sn>self.msn:
            self._bc(sn)
            self.msn=sn
        return (self.cc[:,:,:sn,:].to(i.dtype),self.sc[:,:,:sn,:].to(i.dtype))
def rotary_embedding(q,k,c,s):
    """Applies RoPE to query and key tensors."""
    def hlp(ip):
        i1=ip[...,:ip.shape[-1]//2]
        i2=ip[...,ip.shape[-1]//2:]
        return torch.cat((-i2,i1),dim=-1)
    qe=(q*c)+(hlp(q)*s)
    ke=(k*c)+(hlp(k)*s)
    return qe,ke

Writing rope.py


In [5]:
%%writefile dataset.py
import os
import torch
import sentencepiece as spm
from torch.utils.data import IterableDataset
from args import modelargs
class mydataset(IterableDataset):
    def __init__(self,a:modelargs,dp:str,tp:str):
        """Initializes streaming dataset"""
        super().__init__()
        self.sn=a.max_seq_n
        self.bid=1
        self.eid=2
        self.dp=dp
        self.sp=spm.SentencePieceProcessor(model_file=tp)
        self.lr=int(os.environ.get("LOCAL_RANK","0"))
        self.ws=int(os.environ.get("WORLD_SIZE","1"))
    def __iter__(self):
        """Streams file line-by-line, tokenizes on the fly, and yields chunks."""
        b=[]
        tks=[]
        with open(self.dp,"r",encoding="utf-8") as f:
            for id,l in enumerate(f):
                if id%self.ws!=self.lr:continue
                r=l.strip()
                if len(r)>1:b.append(r)
                if len(b)>=2000:
                    eb=self.sp.Encode(b)
                    for t in eb:
                        tks.append(self.bid)
                        tks.extend(t)
                        tks.append(self.eid)
                    b=[]
                    while len(tks)>=self.sn+1:
                        c=tks[:self.sn+1]
                        tks=tks[self.sn:]
                        yield torch.tensor(c[:-1],dtype=torch.long),torch.tensor(c[1:],dtype=torch.long)

Writing dataset.py


In [6]:
%%writefile dense.py
import torch.nn as nn
from args import modelargs
class SwiGLU(nn.Module):
    def __init__(self,a:modelargs):
        """Initializes dense SwiGLU layers."""
        super().__init__()
        self.w1=nn.Linear(a.dim,a.hidden_dim,bias=False)
        self.w2=nn.Linear(a.hidden_dim,a.dim,bias=False)
        self.w3=nn.Linear(a.dim,a.hidden_dim,bias=False)
        self.silu=nn.SiLU()
    def forward(self,ip):
        """Executes forward pass of dense block."""
        return self.w2(self.silu(self.w1(ip)*self.w3(ip)))

Writing dense.py


In [7]:
%%writefile attention.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from args import modelargs
from rope import rotary_embedding
from cache import KVcache
def rpt_kv(hs:torch.Tensor,nr:int)->torch.Tensor:
    """Repeats key and value states for MQA/GQA."""
    b,nkv,sn,hd=hs.shape
    if nr==1:return hs
    hs=hs[:,:,None,:,:].expand(b,nkv,nr,sn,hd)
    return hs.reshape(b,nkv*nr,sn,hd)
class GQA(nn.Module):
    def __init__(self,a:modelargs):
        """Initializes attention mechanism with QK normalization."""
        super().__init__()
        self.nh=a.n_heads
        self.nkv=a.n_kv_heads
        self.nr=self.nh//self.nkv
        self.hd=a.dim//a.n_heads
        self.wq=nn.Linear(a.dim,self.nh*self.hd,bias=False)
        self.wk=nn.Linear(a.dim,self.nkv*self.hd,bias=False)
        self.wv=nn.Linear(a.dim,self.nkv*self.hd,bias=False)
        self.wo=nn.Linear(self.nh*self.hd,a.dim,bias=False)
        self.qn=nn.RMSNorm(self.hd,eps=a.norm_eps)
        self.kn=nn.RMSNorm(self.hd,eps=a.norm_eps)
    def forward(self,ip:torch.Tensor,cos:torch.Tensor,sin:torch.Tensor,msk:torch.Tensor|None=None,kvc:KVcache|None=None)->torch.Tensor:
        """Executes forward pass calculating attention scores."""
        bn,sn,_=ip.shape
        iq,ik,iv=self.wq(ip),self.wk(ip),self.wv(ip)
        iq=iq.view(bn,sn,self.nh,self.hd)
        ik=ik.view(bn,sn,self.nkv,self.hd)
        iv=iv.view(bn,sn,self.nkv,self.hd).transpose(1,2)
        iq=self.qn(iq).transpose(1,2)
        ik=self.kn(ik).transpose(1,2)
        iq,ik=rotary_embedding(iq,ik,cos,sin)
        if kvc is not None:ik,iv=kvc.change(ik,iv)
        ik=rpt_kv(ik,self.nr)
        iv=rpt_kv(iv,self.nr)
        op=F.scaled_dot_product_attention(iq,ik,iv,attn_mask=msk)
        op=op.transpose(1,2).contiguous().view(bn,sn,-1)
        return self.wo(op)

Writing attention.py


In [8]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from args import modelargs
from rope import RoPE
from attention import GQA
from dense import SwiGLU
from cache import KVcache
class trnsfrmr(nn.Module):
    def __init__(self,a:modelargs):
        """Initializes single transformer block."""
        super().__init__()
        self.an=nn.RMSNorm(a.dim,eps=a.norm_eps)
        self.attn=GQA(a)
        self.fn=nn.RMSNorm(a.dim,eps=a.norm_eps)
        self.lyr=SwiGLU(a)
    def forward(self,ip:torch.Tensor,cos:torch.Tensor,sin:torch.Tensor,msk:torch.Tensor|None=None,kvc:KVcache|None=None)->torch.Tensor:
        """Executes forward pass of transformer block."""
        at=self.attn(self.an(ip),cos,sin,msk,kvc)
        rs=ip+at
        op=rs+self.lyr(self.fn(rs))
        return op
class GIBCS(nn.Module):
    def __init__(self,a:modelargs):
        """Initializes full LLM architecture with weight tying."""
        super().__init__()
        self.a=a
        self.te=nn.Embedding(a.vocab_n,a.dim)
        self.rp=RoPE(a.dim//a.n_heads,a)
        self.ls=nn.ModuleList([trnsfrmr(a) for _ in range(a.n_layers)])
        self.rn=nn.RMSNorm(a.dim,eps=a.norm_eps)
        self.op=nn.Linear(a.dim,a.vocab_n,bias=False)
        self.op.weight=self.te.weight
        self.apply(self._init_w)
    def _init_w(self,m):
        """Initializes weights natively for from-scratch training."""
        if isinstance(m,nn.Linear):
            torch.nn.init.normal_(m.weight,mean=0.0,std=0.02)
            if m.bias is not None:torch.nn.init.zeros_(m.bias)
        elif isinstance(m,nn.Embedding):
            torch.nn.init.normal_(m.weight,mean=0.0,std=0.02)
    def forward(self,tkns:torch.Tensor,tgts:torch.Tensor|None=None,kvcs:list[KVcache]|None=None)->tuple[torch.Tensor,torch.Tensor|None]:
        """Executes full forward pass computing logits and loss."""
        bn,sn=tkns.shape
        t=self.te(tkns)
        sp=kvcs[0].seq_n if kvcs is not None else 0
        cos,sin=self.rp(t,sn+sp)
        cos=cos[:,:,sp:sp+sn,:]
        sin=sin[:,:,sp:sp+sn,:]
        msk=None
        if sn>1:msk=torch.triu(torch.full((sn,sn),float("-inf"),device=tkns.device),diagonal=1)
        for i,l in enumerate(self.ls):
            c=kvcs[i] if kvcs is not None else None
            t=l(t,cos,sin,msk,c)
        lts=self.op(self.rn(t))
        ls=None
        if tgts is not None:ls=F.cross_entropy(lts.view(-1,self.a.vocab_n),tgts.view(-1))
        return lts,ls

Writing model.py


In [9]:
%%writefile train.py
import os
import time
import json
import torch
import matplotlib.pyplot as plt
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from contextlib import nullcontext
from args import modelargs
from model import GIBCS
from dataset import mydataset
def st_ddp():
    """Initializes distributed data parallel environment."""
    os.environ["OMP_NUM_THREADS"]="1"
    dist.init_process_group(backend="nccl")
    lr=int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(lr)
    return lr
def sv_plt(lsh,lhp,lp):
    """Saves loss history array and generates training curve plot."""
    with open(lhp,"w") as f:json.dump(lsh,f)
    plt.figure(figsize=(10,5))
    plt.plot(lsh,label="Train Loss",color="blue",linewidth=1.5)
    plt.title("GIBCS 50M Training Curve")
    plt.xlabel("Logging Steps (x10 Batches)")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()
    plt.savefig(lp)
    plt.close()
def train():
    """Executes training loop with time limit, checkpointing, and loss tracking."""
    st=time.time()
    tl=9.5*3600
    lr=st_ddp()
    d=torch.device(f"cuda:{lr}")
    a=modelargs()
    m=GIBCS(a).to(d)
    if lr==0:print("Compiling model ",flush=True)
    m=torch.compile(m)
    m=DDP(m,device_ids=[lr])
    tp="/kaggle/input/datasets/rishabhtejrajjain/gibcsv2/tokenizer.model"
    dp="/kaggle/input/datasets/rishabhtejrajjain/gibcsv2/train_corpus.txt"
    rcp="/kaggle/input/gibcs-ckpt/gibcs_chckpt.pt"
    cp="/kaggle/working/gibcs_chckpt.pt"
    rlhp="/kaggle/input/gibcs-ckpt/loss_hist.json"
    lhp="/kaggle/working/loss_hist.json"
    lp="/kaggle/working/loss_curve.png"
    lsh=[]
    if lr==0:
        if os.path.exists(rlhp):
            with open(rlhp,"r") as f:lsh=json.load(f)
        elif os.path.exists(lhp):
            with open(lhp,"r") as f:lsh=json.load(f)
    if os.path.exists(rcp):
        if lr==0:print(f"Resuming from {rcp}")
        chk=torch.load(rcp,map_location=d,weights_only=True)
        m.module.load_state_dict(chk['model_state'])
    elif os.path.exists(cp):
        if lr==0:print(f"Resuming from {cp}")
        chk=torch.load(cp,map_location=d,weights_only=True)
        m.module.load_state_dict(chk['model_state'])
    ds=mydataset(a,dp=dp,tp=tp)
    bs=16
    acn=8
    dl=DataLoader(ds,batch_size=bs,num_workers=0,pin_memory=True)
    opt=torch.optim.AdamW(params=m.parameters(),lr=0.0005,weight_decay=0.01,fused=True)
    scl=torch.amp.GradScaler()
    # number of epochs = 1
    m.train()
    for bid,(i,j) in enumerate(dl):
        i,j=i.to(d),j.to(d)
        gac=(bid+1)%acn!=0
        gsy=m.no_sync() if gac else nullcontext()
        with gsy:
            with torch.amp.autocast(device_type="cuda",dtype=torch.float16):
                lts,ls=m(i,j)
                ls=ls/acn
            scl.scale(ls).backward()
        if not gac:
            scl.step(opt)
            scl.update()
            opt.zero_grad()
        if lr==0 and bid%10==0:
            tls=ls.item()*acn
            print(f"BATCH:{bid}, LOSS:{tls:.4f}")
            lsh.append(tls)
        if bid%50==0:
            stp=torch.tensor(0,device=d)
            if lr==0 and (time.time()-st)>tl:stp+=1
            dist.broadcast(stp,src=0)
            if stp.item()==1:
                if lr==0:
                    print("Time limit reached. Saving weights and plot...")
                    torch.save({'model_state':m.module.state_dict()},cp)
                    sv_plt(lsh,lhp,lp)
                dist.destroy_process_group()
                return
if __name__=="__main__":
    train()

Writing train.py


In [10]:
!torchrun --nproc_per_node=2 train.py

W0904 13:18:58.091000 49 torch/distributed/run.py:852] 
W0904 13:18:58.091000 49 torch/distributed/run.py:852] *****************************************
W0904 13:18:58.091000 49 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0904 13:18:58.091000 49 torch/distributed/run.py:852] *****************************************
Compiling model 
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/utils.py:3694: UserWarning: Mismatch dtype between input and weight: input dtype = c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return node.target(*args, **kwargs)  # type: ignore[operator]
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/utils.py:3694: UserWarning: Mismatch dtype between input and we